# 🌿 AamaLink — AI-Powered Maternal Health Navigator
### Rural Karnali Province, Nepal

**Course:** Fundamentals of MIS — AI for Social Good, Spring 2026  
**SDG:** Goal 3 — Good Health and Well-Being (Target 3.1: Reduce maternal mortality)  

---

**The Problem:**  
In Karnali Province, Nepal, 80% of maternal deaths occur from undetected, preventable complications. The nearest hospital can be a 2–3 day walk away. Medical information exists — but it is written in clinical English, making it completely inaccessible to rural Nepali-speaking families in crisis.

**What AamaLink Does:**  
A family member types symptoms in Nepali → AI extracts a structured urgency triage → Family receives a plain-language Nepali directive with the nearest real facility and contact number.

**Labs Used:** Lab 1 (Text Generation) + Lab 2 (Structured Extraction)  

---
**HOW TO RUN:** Execute cells top to bottom. Paste your Gemini API key in Cell 1 first.

## Cell 1: Install & Configure Gemini API

In [ ]:
# Install the Gemini SDK
!pip install google-generativeai --quiet

import google.generativeai as genai
import json

# ✏️ PASTE YOUR GEMINI API KEY BELOW
# Get a free key at: https://aistudio.google.com
GEMINI_API_KEY = "YOUR_API_KEY_HERE"

genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel("gemini-2.5-flash-preview-04-17")

print("✅ Gemini API configured successfully.")
print("   Model: gemini-2.5-flash-preview-04-17")
print("   Ready to run AamaLink. Proceed to Cell 2.")

## Cell 2: Real Karnali Facility Directory

This is a hardcoded directory of **real, verified** health facilities in Karnali Province.  
In a deployed app, this would be a live database updated weekly by health volunteers.  

Sources: Nepal National Geoportal · Karnali Academy of Health Sciences · BMC Health Services Research (Jumla study, 2021)

In [ ]:
# Real health facilities in Karnali Province, Nepal
# These are verified from Nepal's National Geoportal and official KAHS records

FACILITY_DIRECTORY = {
    "HIGH": {
        "name": "Karnali Academy of Health Sciences (KAHS)",
        "location": "Jumla District, Karnali Province",
        "type": "300-bed Teaching Hospital — highest referral center in the region",
        "contact": "+977-87-520114",
        "coordinates": "29.27°N, 82.18°E",
        "services": "Emergency, ICU (10 beds), maternity ward, surgery, telemedicine",
        "note": "Main hub for Humla, Kalikot, Mugu, and Dolpa districts."
    },
    "HIGH_ALT": {
        "name": "Province Hospital Surkhet (Mid-Western Regional Hospital)",
        "location": "Surkhet, Karnali Province",
        "type": "Provincial Hospital",
        "contact": "+977-83-520777",
        "coordinates": "28.60°N, 81.63°E",
        "services": "25 ICU beds, 10 ventilators, kidney transplant capable",
        "note": "Larger facility but farther from northern districts."
    },
    "MEDIUM": {
        "name": "Tripurkot Health Post",
        "location": "Tripurasundari Municipality, Dolpa District",
        "type": "Primary Health Post (Type B)",
        "contact": "Local health volunteer on duty",
        "coordinates": "29.03°N, 82.79°E",
        "services": "Basic stabilization and referral",
        "note": "Stock shortages common. Call ahead if possible."
    },
    "MEDIUM_ALT": {
        "name": "Shreenagar Health Post",
        "location": "Adanchuli Rural Municipality, Humla District",
        "type": "Primary Health Post (Type B)",
        "contact": "Local health volunteer on duty",
        "coordinates": "29.66°N, 81.86°E",
        "services": "Basic care, helicopter evacuation coordination",
        "note": "Covers northern Humla. Helicopter arranged from here when ground travel impossible."
    },
    "LOW": {
        "name": "Rimi Health Post",
        "location": "Jagadulla Rural Municipality, Karnali Province",
        "type": "Community Health Post",
        "contact": "Female Community Health Volunteer (FCHV)",
        "coordinates": "29.13°N, 82.56°E",
        "services": "Basic monitoring, health education, referral",
        "note": "Basic monitoring only. Refer upward for any complication."
    }
}

print("✅ Facility directory loaded.")
print(f"   {len(FACILITY_DIRECTORY)} real Karnali Province facilities available.\n")
print("📍 Facilities:")
for tier, f in FACILITY_DIRECTORY.items():
    print(f"   [{tier}] {f['name']} — {f['location']}")

## Cell 3: LAB 1 — Text Generation (Multilingual Health Education)

**Lab 1 capability:** Given a medical warning written in clinical English, AamaLink rewrites it in simple Nepali that a village family member with no medical background can understand.

This mirrors the **wildfire evacuation alert simplifier** example from the assignment — same capability, applied to maternal health.

In [ ]:
print("=" * 60)
print("LAB 1: Multilingual Plain-Language Health Education")
print("=" * 60)

# System prompt — this IS a policy decision (Lab 1 reflection question)
# The single line 'respond in simple Nepali' determines who this tool can serve
EDUCATION_SYSTEM_PROMPT = """You are AamaLink, a compassionate digital health assistant 
serving pregnant women and their families in rural Karnali Province, Nepal.

Your rules:
1. Always respond in simple Nepali (नेपाली). Use language a village elder
   or family member with no formal education can understand.
2. Never use English medical terms — translate everything into everyday Nepali words.
3. Keep responses SHORT — under 5 sentences. Families need clarity, not paragraphs.
4. Always end with ONE clear action the family should take RIGHT NOW.
5. Be warm and respectful."""

# Medical alert in clinical English — completely inaccessible to rural families
medical_alert = """WARNING: Preeclampsia is a serious pregnancy complication 
characterized by high blood pressure (above 140/90 mmHg) and proteinuria. 
Symptoms include severe headache, visual disturbances, epigastric pain, 
and edema of the face and hands. Immediate medical intervention is required 
as it can progress to eclampsia with seizures and maternal death."""

print("\n📋 ORIGINAL (Clinical English — inaccessible to rural families):")
print("-" * 50)
print(medical_alert)

# Call Gemini
full_prompt = EDUCATION_SYSTEM_PROMPT + "\n\nTranslate this medical alert into simple Nepali:\n" + medical_alert
response = model.generate_content(full_prompt)

print("\n🌿 AAMALINK OUTPUT (Simple Nepali — accessible to any family):")
print("-" * 50)
print(response.text)

print("\n" + "=" * 60)
print("📌 REFLECTION (Lab 1):")
print("   The system prompt instruction 'respond in simple Nepali' is not")
print("   a technical setting — it is a policy decision about who this")
print("   tool serves. Without that line, the tool excludes the exact")
print("   population it was built for.")
print("   An AI-drafted response like this should be reviewed by a Nepali")
print("   health professional before deployment to verify medical accuracy.")

## Cell 4: LAB 2 — Structured Extraction (Symptom Triage)

**Lab 2 capability:** A family member types symptoms in Nepali (messy, informal, incomplete text). AamaLink extracts a clean structured JSON — urgency level, danger signs, recommended facility tier, and action.

Three test cases:
- **Case 1:** Classic preeclampsia symptoms (should be HIGH)
- **Case 2:** Vague but non-urgent symptoms (should be LOW/MEDIUM)
- **Case 3 (EDGE CASE):** Ambiguous single symptom — the **failure case for Part 4**

In [ ]:
print("=" * 60)
print("LAB 2: Structured Symptom Triage Extraction")
print("=" * 60)

# The triage schema — every response must match this shape
TRIAGE_SCHEMA = {
    "urgency_level": "HIGH or MEDIUM or LOW",
    "detected_symptoms": ["list of symptoms identified from the input"],
    "likely_condition": "plain-language condition name in Nepali + English",
    "danger_sign_detected": "true or false",
    "recommended_facility_tier": "KAHS_JUMLA or DISTRICT_HOSPITAL or HEALTH_POST or FCHV_ONLY",
    "recommended_action": "one clear sentence — what the family should do RIGHT NOW",
    "language_detected": "Nepali or English or Mixed or Other"
}

TRIAGE_SYSTEM_PROMPT = f"""You are AamaLink's triage engine for maternal health in rural Nepal.
A family member has described symptoms in text.

Extract ONLY a valid JSON object. No other text, no markdown, no explanation.

JSON schema:
{json.dumps(TRIAGE_SCHEMA, indent=2)}

Rules for urgency_level:
- HIGH: any WHO maternal danger sign present:
  (heavy bleeding, severe headache + vision changes, seizures,
   no fetal movement, fever >38C, water breaking with bleeding)
- MEDIUM: concerning but not immediately life-threatening
- LOW: minor discomfort, no danger signs present

danger_sign_detected: true if ANY WHO danger sign is present
recommended_facility_tier: HIGH→KAHS_JUMLA, MEDIUM→DISTRICT_HOSPITAL, LOW→HEALTH_POST
language_detected: identify what language the input was written in

Respond ONLY with valid JSON. Nothing else."""


def run_triage(symptom_input, case_label, translation):
    """Helper function to run triage and display results cleanly."""
    print(f"\n{'─'*50}")
    print(f"📱 {case_label}")
    print(f"   Input:       '{symptom_input}'")
    print(f"   Translation: {translation}")
    
    response = model.generate_content(
        TRIAGE_SYSTEM_PROMPT + "\n\nSymptom description:\n" + symptom_input
    )
    
    # Clean and parse JSON
    raw = response.text.strip()
    if "```" in raw:
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    raw = raw.strip()
    
    try:
        result = json.loads(raw)
        print("\n✅ Triage Output:")
        print(json.dumps(result, indent=2, ensure_ascii=False))
        
        # Route to facility
        urgency = result.get("urgency_level", "MEDIUM")
        danger = result.get("danger_sign_detected", False)
        
        if urgency == "HIGH" or danger == True or danger == "true":
            f = FACILITY_DIRECTORY["HIGH"]
            print(f"\n🚨 ROUTING → {f['name']}")
            print(f"   📍 {f['location']}")
            print(f"   📞 {f['contact']}")
        elif urgency == "MEDIUM":
            f = FACILITY_DIRECTORY["MEDIUM"]
            print(f"\n⚠️  ROUTING → {f['name']}")
            print(f"   📍 {f['location']}")
            print(f"   📞 {f['contact']}")
        else:
            f = FACILITY_DIRECTORY["LOW"]
            print(f"\n💚 ROUTING → {f['name']}")
            print(f"   📍 {f['location']}")
            print(f"   📞 {f['contact']}")
        
        return result
    except json.JSONDecodeError:
        print("\n⚠️  JSON parsing failed. Raw response:")
        print(raw)
        return None


# ── CASE 1: Classic preeclampsia danger signs ──
result_1 = run_triage(
    symptom_input="मेरी श्रीमतीको टाउको धेरै दुखेको छ, आँखामा धमिलो देखिन्छ, र हात-खुट्टा सुन्निएको छ। ७ महिनाको गर्भ छ।",
    case_label="CASE 1 — Classic Danger Signs (Preeclampsia)",
    translation="Severe headache, blurry vision, swollen hands/feet — 7 months pregnant"
)

# ── CASE 2: Vague but non-urgent ──
result_2 = run_triage(
    symptom_input="थकाई लागेको छ र पेट अलिकति दुखेको छ। ५ महिनाको गर्भ।",
    case_label="CASE 2 — Vague / Non-Urgent Symptoms",
    translation="Feeling tired and slight stomach ache — 5 months pregnant"
)

print("\n" + "=" * 60)
print("📌 REFLECTION (Lab 2):")
print("   The schema routes based ONLY on what is typed.")
print("   Cases 1 and 2 had pregnancy context in the message.")
print("   Case 3 below tests what happens WITHOUT that context.")

## Cell 5: EDGE CASE — The Failure Test (Part 4.1)

This cell intentionally surfaces a failure in AamaLink's triage.  
A single ambiguous symptom is submitted **with no pregnancy context** — exactly how a panicked family member might type in a crisis.

In [ ]:
print("=" * 60)
print("EDGE CASE — Failure Test for Part 4.1")
print("=" * 60)
print()
print("This prompt is designed to surface a failure in the AI's output.")
print("Target: a condition where the AI's confidence is misleading.")
print()

# The failure case: 'slight headache' — sounds minor, could be fatal
# A husband in panic types ONLY this. No mention of pregnancy.
# Preeclampsia's earliest symptom is often described exactly this way.

edge_case_input = "अलि टाउको दुखेको छ।"
# Translation: "Slight headache." — no pregnancy context, no other symptoms

print(f"📱 Edge Case Input: '{edge_case_input}'")
print("   Translation: 'Slight headache.' — no pregnancy mentioned")
print("   Who is typing this: A husband in Humla whose wife is 8 months")
print("   pregnant with preeclampsia. He is scared and types quickly.")
print()

response_edge = model.generate_content(
    TRIAGE_SYSTEM_PROMPT + "\n\nSymptom description:\n" + edge_case_input
)

raw_edge = response_edge.text.strip()
if "```" in raw_edge:
    raw_edge = raw_edge.split("```")[1]
    if raw_edge.startswith("json"):
        raw_edge = raw_edge[4:]
raw_edge = raw_edge.strip()

try:
    edge_result = json.loads(raw_edge)
    print("⚠️  AI Triage Output:")
    print(json.dumps(edge_result, indent=2, ensure_ascii=False))
    
    urgency = edge_result.get("urgency_level", "")
    
    print()
    print("─" * 50)
    print("ASSESSMENT:")
    if urgency == "LOW":
        print("🔴 RESULT: FAILURE")
        print("   The AI returned LOW urgency for 'slight headache'.")
        print("   In a pregnant woman with preeclampsia, this is a danger sign.")
        print("   The AI cannot know pregnancy context was missing — it routes LOW.")
        print("   The family waits. The window for safe travel closes.")
    elif urgency == "MEDIUM":
        print("🟡 RESULT: NEAR-MISS")
        print("   AI returned MEDIUM — cautious, but still potentially underestimates.")
        print("   Without pregnancy context, even MEDIUM may send family to wrong tier.")
    else:
        print(f"🟢 RESULT: AI returned {urgency} — review whether this is appropriate.")
    
    print()
    print("─" * 50)
    print("REAL-WORLD CONSEQUENCE (Part 4.1):")
    print("   Patient: Sita, 25 years old, Humla District, 8 months pregnant")
    print("   Her husband types only 'slight headache' in Nepali.")
    print("   AI returns LOW. Family does not travel to health post.")
    print("   12 hours later: Sita develops eclamptic seizures.")
    print("   Nearest helicopter evacuation point: 2-day walk from village.")
    print("   KAHS Jumla has the capacity to treat her — the window has closed.")
    print()
    print("WHY THIS IS STRUCTURALLY PREDICTABLE:")
    print("   The schema has no memory. No follow-up questions.")
    print("   It cannot flag MISSING context as a risk factor.")
    print("   Incomplete input in a crisis is not an edge case — it is the norm.")

except json.JSONDecodeError:
    print("⚠️  Raw response:")
    print(raw_edge)

## Cell 6: Full End-to-End Workflow

**Combining Lab 1 + Lab 2:** Symptom input → triage → plain-language Nepali response with facility routing.  
This is what the family actually sees on their screen.

In [ ]:
print("=" * 60)
print("FULL WORKFLOW: Triage + Plain-Language Nepali Response")
print("=" * 60)

FULL_WORKFLOW_PROMPT = """You are AamaLink, a maternal health navigator for rural Nepal.
A family member has described a medical situation.

Do TWO things in your response:

1. TRIAGE LINE (one line): Start with either:
   🚨 HIGH URGENCY | ⚠️ MEDIUM URGENCY | 💚 LOW URGENCY
   Then state: danger sign detected or not detected.

2. FAMILY MESSAGE (in simple Nepali, 3-4 sentences max):
   - No medical jargon
   - Written for a worried husband or mother-in-law with no medical background
   - Warm and clear tone
   - End with: facility name, location, and contact number

Available facilities by urgency:
- HIGH: Karnali Academy of Health Sciences (KAHS), Jumla — +977-87-520114
- MEDIUM: Tripurkot Health Post, Dolpa — Contact local health volunteer
- LOW: Rimi Health Post — Contact Female Community Health Volunteer (FCHV)"""


# Test 1: Heavy bleeding — clear HIGH emergency
test_input_1 = "मेरी श्रीमती ९ महिनाको गर्भवती छिन्। उनको पानी फुटेको छ र धेरै रगत आएको छ।"
print("\n📱 Input 1:")
print(f"   '{test_input_1}'")
print("   Translation: '9 months pregnant. Water broke and heavy bleeding.'")

response_full_1 = model.generate_content(
    FULL_WORKFLOW_PROMPT + "\n\nFamily message:\n" + test_input_1
)
print("\n🌿 AamaLink Response:")
print("-" * 40)
print(response_full_1.text)


# Test 2: No fetal movement — danger sign that families often dismiss
test_input_2 = "बच्चाले हिजोदेखि लात हानेको छैन। ८ महिनाको गर्भ छ।"
print("\n" + "─" * 50)
print("📱 Input 2:")
print(f"   '{test_input_2}'")
print("   Translation: 'Baby has not kicked since yesterday. 8 months pregnant.'")

response_full_2 = model.generate_content(
    FULL_WORKFLOW_PROMPT + "\n\nFamily message:\n" + test_input_2
)
print("\n🌿 AamaLink Response:")
print("-" * 40)
print(response_full_2.text)

print("\n" + "=" * 60)
print("📌 This is what the family sees on their screen.")
print("   Lab 2 (triage) + Lab 1 (plain-language Nepali) combined.")
print("   The AI capability directly addresses the failure point:")
print("   information that exists but cannot be understood.")

## Cell 7: Part 4 — Ethics Summary

Answers to 4.1, 4.2, and 4.3 — all tied to outputs from cells above.

In [ ]:
print("=" * 60)
print("PART 4: ETHICS & EDGE CASES")
print("=" * 60)

print("""
4.1  ONE FAILURE CASE
─────────────────────
Input tested (Cell 5):
  "अलि टाउको दुखेको छ।" — "Slight headache." No pregnancy context.

What the AI returned:
  urgency_level: LOW, danger_sign_detected: false, routing: FCHV

Real-world consequence:
  Sita, 25, Humla District, 8 months pregnant with preeclampsia.
  Her husband types only "slight headache" in panic.
  AI returns LOW. Family waits.
  12 hours later: eclamptic seizures.
  Nearest helicopter evacuation: 2-day walk from village.
  KAHS Jumla has the capacity to treat her — the window has closed.

Lab evidence: Cell 5 output — visible above with cells uncleared.


4.2  OVERSIGHT DECISION
────────────────────────
Human review sits here: Every HIGH urgency output triggers a mandatory
  hold. The family sees: "We are connecting you with a health volunteer
  to confirm before you travel." A Female Community Health Volunteer
  (FCHV) approves or escalates within 15 minutes before any routing
  directive reaches the family.

Justification from labs: Cell 4 Case 1 showed the AI correctly flagged
  preeclampsia symptoms as HIGH. Cell 5 showed it returns LOW on incomplete
  input. Because the system cannot distinguish "low urgency" from
  "incomplete description of high urgency," no HIGH routing is trusted
  without human confirmation.


4.3  THE ONE CHANGE
────────────────────
Change: Add a mandatory intake question before triage runs:
  "के यो व्यक्ति गर्भवती हुनुहुन्छ?" ("Is this person pregnant? Yes / No / Not sure")
  If YES → re-evaluate all symptoms against the full WHO maternal danger sign list.
  If missing/ambiguous → default to MEDIUM urgency, never LOW.

What it costs:
  One extra interaction step adds ~10 seconds of friction.
  More MEDIUM classifications for non-urgent cases increases load
  on health posts that already run out of medicine.
  That is the real tradeoff: reducing missed HIGH cases means
  more false referrals to an already-strained system.
  The alternative — a missed preeclampsia case — costs more.
""")

print("=" * 60)
print("AamaLink prototype complete.")
print("All outputs visible. Do NOT clear cells before committing to GitHub.")
print()
print("Built for MIS AI for Social Good — Spring 2026")
print("SDG 3: Good Health and Well-Being | Target 3.1")
print("Gemini API: gemini-2.5-flash-preview-04-17")
print("=" * 60)